## SyntheticIdentityDetection - XGBoost

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from snowflake.snowpark import Session
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully")
print("Note: Using class weights instead of SMOTE for handling class imbalance")

In [ ]:
# Get Snowpark session (available in Snowflake Notebooks)
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print(f"Current database: {session.get_current_database()}")
print(f"Current schema: {session.get_current_schema()}")
print(f"Current warehouse: {session.get_current_warehouse()}")
print(f"Current role: {session.get_current_role()}")

In [ ]:
# Load the feature table
query = """
SELECT * 
FROM A01A0E_GBU_FINCRIME_POC.GRAPH_ONTOLOGY.FRAUD_ML_FEATURES
"""

df_snowpark = session.sql(query)
print(f"Data loaded from FRAUD_ML_FEATURES")
print(f"Total rows: {df_snowpark.count()}")

# Convert to Pandas for local processing
df = df_snowpark.to_pandas()
print(f"\nConverted to Pandas DataFrame: {df.shape}")

#### Data Analysis

In [ ]:
# Display basic info
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"Shape: {df.shape}")
print(f"\nColumn count: {len(df.columns)}")
print(f"\nTarget distribution:")
print(df['TARGET'].value_counts())
print(f"\nFraud rate: {df['TARGET'].sum() / len(df) * 100:.2f}%")

# Check for missing values
print(f"\nMissing values per column:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0].sort_values(ascending=False))
else:
    print("No missing values found!")

In [ ]:
# Display feature columns
print("\n=" * 80)
print("FEATURE COLUMNS")
print("=" * 80)

# Exclude non-feature columns
exclude_cols = ['TRANSACTION_ID', 'TARGET', 'TRANSACTION_TIME', 'MERCHANT_NAME', 'LOCATION_NAME']
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"\nTotal features: {len(feature_cols)}")
print(f"\nFeature list:")
for i, col in enumerate(feature_cols, 1):
    print(f"{i:2d}. {col}")

#### Feature Selection and Preparation

In [ ]:
# Define feature columns (exclude identifiers, target, and text columns)
exclude_cols = ['TRANSACTION_ID', 'TARGET', 'TRANSACTION_TIME', 'MERCHANT_NAME', 'LOCATION_NAME']
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"Selected {len(feature_cols)} features for training")

# Prepare X and y
X = df[feature_cols].copy()
y = df['TARGET'].astype(int)  # Convert boolean to int

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nTarget distribution in y:")
print(y.value_counts())

In [ ]:
# Handle missing values if any
if X.isnull().sum().sum() > 0:
    print("⚠️ Filling missing values with median...")
    X = X.fillna(X.median())
    print("Missing values handled")
else:
    print("No missing values to handle")

#### Train-Test Split

In [ ]:
# Stratified split to maintain fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("=" * 80)
print("TRAIN-TEST SPLIT")
print("=" * 80)
print(f"\nTraining set: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTraining set fraud: {y_train.sum():,} ({y_train.sum()/len(y_train)*100:.2f}%)")
print(f"Test set fraud: {y_test.sum():,} ({y_test.sum()/len(y_test)*100:.2f}%)")
print("\nStratified split maintains fraud ratio in both sets")

## 6. Handle Class Imbalance with Oversampling

In [ ]:
# Handle class imbalance with Random Oversampling
# (Alternative to SMOTE since imblearn package is not available in Snowflake Notebooks)
print("=" * 80)
print("HANDLING CLASS IMBALANCE WITH RANDOM OVERSAMPLING")
print("=" * 80)

print(f"\nBefore Oversampling:")
print(f"  Legitimate: {(y_train == 0).sum():,}")
print(f"  Fraud: {(y_train == 1).sum():,}")
print(f"  Ratio: 1:{(y_train == 0).sum() / (y_train == 1).sum():.1f}")

# Simple random oversampling of minority class (fraud)
fraud_indices = y_train[y_train == 1].index
legit_indices = y_train[y_train == 0].index

# Calculate how many fraud samples we need (50% of legitimate)
target_fraud_count = len(legit_indices) // 2

# Randomly sample fraud cases with replacement to reach target count
np.random.seed(42)
oversampled_fraud_indices = np.random.choice(fraud_indices, size=target_fraud_count, replace=True)

# Combine legitimate samples with oversampled fraud samples
balanced_indices = np.concatenate([legit_indices, oversampled_fraud_indices])
np.random.shuffle(balanced_indices)

X_train_balanced = X_train.loc[balanced_indices]
y_train_balanced = y_train.loc[balanced_indices]

print(f"\nAfter Oversampling:")
print(f"  Legitimate: {(y_train_balanced == 0).sum():,}")
print(f"  Fraud: {(y_train_balanced == 1).sum():,}")
print(f"  Ratio: 1:{(y_train_balanced == 0).sum() / (y_train_balanced == 1).sum():.1f}")
print(f"\n✅ Training set now balanced with random oversampling")

#### Train XGBoost Model

In [ ]:
# Initialize XGBoost classifier with optimal hyperparameters
print("=" * 80)
print("TRAINING XGBOOST MODEL")
print("=" * 80)

model = xgb.XGBClassifier(
    # Core parameters
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    
    # Handle imbalance (even with SMOTE, helps)
    scale_pos_weight=2,  # Give more weight to fraud class
    
    # Regularization to prevent overfitting
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    
    # Other parameters
    objective='binary:logistic',
    eval_metric='aucpr',  # Precision-Recall AUC for imbalanced data
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1
)

print("\nTraining XGBoost model...")
print(f"Training samples: {X_train_balanced.shape[0]:,}")
print(f"Features: {X_train_balanced.shape[1]}")

# Train the model
model.fit(X_train_balanced, y_train_balanced)

print("\nModel training completed!")

#### Model Evaluation

In [ ]:
# Make predictions on test set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("=" * 80)
print("MODEL EVALUATION RESULTS")
print("=" * 80)

# Confusion Matrix
print("\n1. CONFUSION MATRIX")
print("-" * 40)
cm = confusion_matrix(y_test, y_pred)
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives (TN): {tn:,}")
print(f"False Positives (FP): {fp:,}")
print(f"False Negatives (FN): {fn:,}")
print(f"True Positives (TP): {tp:,}")
print(f"\nFalse Positive Rate: {fp/(fp+tn)*100:.2f}%")
print(f"Fraud Detection Rate (Recall): {tp/(tp+fn)*100:.2f}%")

In [ ]:
# Classification Report
print("\n2. CLASSIFICATION REPORT")
print("-" * 40)
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud'], digits=4))

In [ ]:
# ROC-AUC and PR-AUC scores
print("\n3. PERFORMANCE METRICS")
print("-" * 40)
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Precision-Recall AUC (more important for imbalanced data)
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall, precision)
print(f"Precision-Recall AUC: {pr_auc:.4f}  (Primary metric for imbalanced data)")

# Business metrics
fraud_amount_prevented = tp * 2000  # Assume avg fraud = $2000
false_positive_cost = fp * 10  # Assume investigation cost = $10
net_savings = fraud_amount_prevented - false_positive_cost

print(f"\n4. BUSINESS IMPACT (ESTIMATED)")
print("-" * 40)
print(f"Frauds Detected: {tp}/{tp+fn} ({tp/(tp+fn)*100:.1f}%)")
print(f"Fraud Amount Prevented: ${fraud_amount_prevented:,}")
print(f"False Positive Cost: ${false_positive_cost:,}")
print(f"Net Savings: ${net_savings:,}")
print(f"\n Model shows strong performance on imbalanced fraud data!")

#### Feature Importance Analysis

In [ ]:
# Get feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("=" * 80)
print("TOP 20 MOST IMPORTANT FEATURES")
print("=" * 80)
print(feature_importance.head(20).to_string(index=False))

# Visualize top 15 features
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance Score')
plt.title('Top 15 Most Important Features for Fraud Detection')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nFeature importance analysis complete")

#### Save Model

In [ ]:
# Save the trained model
import joblib
import os
import tempfile

# Use temporary directory (writable in Snowflake Notebooks)
model_dir = '/tmp/models'
os.makedirs(model_dir, exist_ok=True)

model_path = f'{model_dir}/fraud_detection_xgboost.pkl'
joblib.dump(model, model_path)

print("=" * 80)
print("MODEL SAVED")
print("=" * 80)
print(f"\nModel saved to: {model_path}")
print(f"File size: {os.path.getsize(model_path) / 1024:.2f} KB")

# Save feature list for inference
feature_list_path = f'{model_dir}/feature_columns.txt'
with open(feature_list_path, 'w') as f:
    f.write('\n'.join(feature_cols))
print(f"Feature list saved to: {feature_list_path}")
print(f"\nModel ready for deployment!")
print(f"\nNote: Model is saved in /tmp (temporary). For production deployment:")
print(f"  1. Register model to Snowflake Model Registry")
print(f"  2. Or upload to a Snowflake stage for persistence")

#### Real-Time Inference

In [ ]:
# Test real-time inference
print("=" * 80)
print("REAL-TIME FRAUD DETECTION DEMO")
print("=" * 80)

# Get a sample transaction from test set
test_sample_idx = 5
test_transaction = X_test.iloc[test_sample_idx:test_sample_idx+1]
actual_label = y_test.iloc[test_sample_idx]

# Make prediction
prediction = model.predict(test_transaction)[0]
fraud_probability = model.predict_proba(test_transaction)[0, 1]

print(f"\nTransaction Sample #{test_sample_idx}:")
print(f"  Amount: ${test_transaction['AMOUNT'].values[0]:.2f}")
print(f"  Hour: {test_transaction['HOUR_OF_DAY'].values[0]}")
print(f"  Is High Risk Merchant: {bool(test_transaction['IS_HIGH_RISK_MERCHANT'].values[0])}")
print(f"  Is High Risk Location: {bool(test_transaction['IS_HIGH_RISK_LOCATION'].values[0])}")
print(f"\nPrediction:")
print(f"  Fraud Probability: {fraud_probability*100:.2f}%")
print(f"  Predicted: {'FRAUD' if prediction == 1 else 'LEGITIMATE'}")
print(f"  Actual: {'FRAUD' if actual_label == 1 else 'LEGITIMATE'}")
print(f"  Result: {'CORRECT' if prediction == actual_label else 'INCORRECT'}")

# Decision threshold
if fraud_probability > 0.5:
    print(f"\nACTION: Block transaction and trigger manual review")
elif fraud_probability > 0.3:
    print(f"\nACTION: Flag for additional verification")
else:
    print(f"\nACTION: Approve transaction")

## 12. Summary and Next Steps

### Model Performance Summary:
- **Precision-Recall AUC**: Primary metric for imbalanced fraud detection
- **ROC-AUC**: Overall model discrimination
- **Fraud Detection Rate**: Percentage of frauds caught
- **False Positive Rate**: Legitimate transactions incorrectly flagged

### Not Done - But have been trialed before:
1. **Register Model**: Save to Snowflake Model Registry for version control
2. **Deploy for Real-Time Inference**: Create Snowpark Container Service endpoint
3. **Monitor Model**: Set up drift detection and performance tracking
4. **A/B Testing**: Compare with baseline or other models
5. **Iterate**: Retrain with new data, tune hyperparameters, add features

### Deployment Options:
- **Batch Inference**: Process large volumes with `mv.run_batch()`
- **Real-Time API**: Deploy SPCS inference service with REST endpoint
- **SQL Integration**: Embed predictions in SQL pipelines